# What's new in `unify-compilation-workflow`

A chipper tour of everything new on this branch — the unified
compilation workflow that turns AIE designs into Triton-style JIT
functions, packaged kernel factories, content-addressed caching,
tensor-arg validation, runtime/trace tooling, and the bench/verify
helpers that go with them.

Every cell here runs on a Phoenix NPU with `ironenv` activated.

> **Heads up on warnings**: every `@iron.jit` design with an unbound
> `Compile[T]` parameter emits a friendly `UserWarning` reminding you
> to pass it at every call. Those are *informational*, not errors —
> they show up several times in the cells below.

**TL;DR — what's new:**

1. `@iron.jit` decorator — one Python function = one JIT'd design.
2. `aie.iron.kernels.*` factories — packaged C++ kernel wrappers (no more hand-rolled `Kernel(...)` + Makefile rules).
3. `Compile[T]` / `In` / `Out` / `InOut` annotation markers + 3 ways to bind compile params.
4. `CompilableDesign` / `CallableDesign` API: `lower()` / `specialize()` / `compile()` / `__call__`.
5. Content-addressed kernel cache in `~/.npu/cache/`, with hash invalidation on Peano upgrade *and* Peano-absent tolerance.
6. Tensor-arg validation with multi-DMA fan-out support.
7. `aie.utils.benchmark` (`run_iters`, `print_benchmark`).
8. `aie.utils.verify` (`nearly_equal`, `count_mismatches`).
9. `aie.utils.trace` integration: TraceConfig + JIT + `trace_to_json` + `print_cycles_summary`.
10. Compile-process logging: every clang/aiecc command goes through `logging.getLogger("aie.utils.compile")`.

## 0. Setup

Pick a device and import the iron surface.

In [1]:
import logging
import time
from pathlib import Path

import numpy as np

import aie.iron as iron
from aie.iron import (
    Compile,
    In,
    Out,
    ObjectFifo,
    Program,
    Runtime,
    Worker,
    jit,
    kernels,
)
from aie.iron.controlflow import range_

# No need to call set_current_device — iron.get_current_device() falls back
# to DefaultNPURuntime.device(), which queries XRT for the actual hardware
# (NPU1 = Phoenix, NPU2 = Strix). Both `lower()` and on-NPU calls use it.
device = iron.get_current_device()
print("Auto-detected device:", type(device).__name__)

Auto-detected device: NPU1


## 1. `@iron.jit` — Triton-style design compilation

The decorator turns a Python function that *generates* an MLIR module
into something you call like a normal function. The body runs at
*compile* time; the returned `Program(...).resolve_program()` becomes
the cached kernel.

A minimal passthrough fits in 20 lines:

In [2]:
@jit
def passthrough(x: In, y: Out, *, N: Compile[int]):
    line_ty = np.ndarray[(N,), np.dtype[np.uint8]]

    of_in = ObjectFifo(line_ty, name="in")
    of_out = ObjectFifo(line_ty, name="out")

    def core_fn(of_in, of_out):
        elem_in = of_in.acquire(1)
        elem_out = of_out.acquire(1)
        for i in range_(N):
            elem_out[i] = elem_in[i]
        of_in.release(1)
        of_out.release(1)

    worker = Worker(core_fn, [of_in.cons(), of_out.prod()])
    rt = Runtime()
    with rt.sequence(line_ty, line_ty) as (a_in, b_out):
        rt.start(worker)
        rt.fill(of_in.prod(), a_in)
        rt.drain(of_out.cons(), b_out, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


# `passthrough` is now a CallableDesign, not a function.
print(repr(passthrough))

CallableDesign(CompilableDesign(generator='passthrough', compile_kwargs={}))


/tmp/ipykernel_64660/3998335787.py:2: UserWarning: 'passthrough' has Compile[T] parameters with no defaults and no pre-bound values: ['N'].
  You must pass these as keyword arguments at every call:
    kernel(..., N=...)
  Omitting them will raise TypeError at compile time.
  def passthrough(x: In, y: Out, *, N: Compile[int]):


### Lower it (cheap) — generate MLIR without the slow aiecc step

`lower()` runs the generator and returns the MLIR module as a string.
Useful for sanity-checking the structure before the full compile.

In [3]:
mlir = passthrough.lower(N=4096)
print("\n".join(mlir.splitlines()[:6]))
print(f"... ({len(mlir.splitlines())} lines total)")

module {
  aie.device(npu1) {
    %logical_core = aie.logical_tile<CoreTile>(?, ?)
    %logical_shim_noc = aie.logical_tile<ShimNOCTile>(?, ?)
    %logical_shim_noc_0 = aie.logical_tile<ShimNOCTile>(?, ?)
    aie.objectfifo @in(%logical_shim_noc, {%logical_core}, 2 : i32) : !aie.objectfifo<memref<4096xui8>> 
... (44 lines total)


### Run it (compiles to xclbin on first call, then cached)

Calling the design with `iron.tensor` inputs JIT-compiles
(if not already cached) and runs on the NPU.

In [4]:
N = 4096
in_np = np.arange(N, dtype=np.uint8)
in_t = iron.tensor(in_np, dtype=np.uint8)
out_t = iron.zeros(N, dtype=np.uint8)

passthrough(in_t, out_t, N=N)
assert np.array_equal(in_t.numpy(), out_t.numpy())
print(f"PASS — {N} bytes round-tripped through the NPU.")

PASS — 4096 bytes round-tripped through the NPU.


## 2. The kernel library: `aie.iron.kernels`

Before this branch, every design hand-rolled `Kernel("conv2dk1_i8",
"conv2dk1_i8.o", [int8_input_ty, ...])` declarations *and* maintained
a Makefile rule that compiled the `.cc` to `.o` with the right
`-D{INT8,UINT8}_ACT` flag.

The kernel library packages all of that — source path, compile flags,
typed argument list — behind one factory call.

In [5]:
conv = kernels.conv2dk1(
    input_width=32,
    input_channels=64,
    output_channels=64,
    act_dtype=np.int8,
)
# `arg_types` is public.  The other introspection attributes
# (`_name`, `_source_file`, `_compile_flags`) are still private — we peek
# at them here purely to show what the factory packaged for you.
print(f"function name : {conv._name}")
print(f"source        : {Path(conv._source_file).name}")
print(f"compile flags : {conv._compile_flags}")
print(f"arg count     : {len(conv.arg_types())}")
for i, t in enumerate(conv.arg_types()[:3]):
    print(f"  arg[{i}] : {t}")
print("  ...")

function name : conv2dk1_i8
source        : conv2dk1.cc
compile flags : ['-DINT8_ACT']
arg count     : 7
  arg[0] : numpy.ndarray[(2048,), numpy.dtype[numpy.int8]]
  arg[1] : numpy.ndarray[(4096,), numpy.dtype[numpy.int8]]
  arg[2] : numpy.ndarray[(2048,), numpy.dtype[numpy.uint8]]
  ...


The library covers conv variants, eltwise, reductions, activations
(ReLU, GeLU, SwiGLU, softmax), vision (filter2d, RGBA conversions),
matmul (`mm`, `mm_zero`, `cascade_mm`, `mv`), LUT-backed `bf16_exp`,
and bottleneck conv combos:

In [6]:
import aie.iron.kernels as K

named = sorted(n for n in dir(K) if not n.startswith("_") and not n[0].isupper())
factories = [
    n
    for n in named
    if n not in ("conv", "eltwise", "linalg", "reduce", "vision", "activation")
]
print(f"{len(factories)} factories available:")
for i in range(0, len(factories), 6):
    print("  " + ", ".join(factories[i : i + 6]))

36 factories available:
  add, add_weighted, bf16_exp, bitwise_and, bitwise_or, bn_conv2dk1_i8
  bn_conv2dk1_relu, bn_conv2dk1_skip, bn_conv2dk3, bn_conv2dk3_dw, cascade_mm, conv2dk1
  conv2dk14, conv2dk1_i8, conv2dk1_skip, conv2dk1_skip_init, conv2dk3, filter2d
  gelu, gray2rgba, mm, mm_zero, mul, mv
  passthrough, reduce_add, reduce_max, reduce_min, relu, rgba2gray
  rgba2hue, scale, silu, softmax, swiglu, threshold


### New on this branch: shared-buffer parameters

Two factories grew optional knobs for designs that share weight buffers
across multiple workers — both motivated by the resnet conv2_x port.

- **`kernels.conv2dk1_skip_init(skip_input_channels=...)`** — sizes the
  weights buffer for inline skip-projection weights and the skip buffer
  for the unprojected channel count.
- **`kernels.conv2dk3(weight_output_channels=...)`** — sizes the weights
  buffer for *more* output channels than a single call produces; multiple
  workers share one buffer and pick their slice via `channel_offset`.

In [7]:
def wt_skip(k):
    return k.arg_types()[2].__args__[0][0], k.arg_types()[4].__args__[0][0]


# Default: skip_input_channels falls back to input_channels.
default = kernels.conv2dk1_skip_init(input_channels=64, output_channels=256)
print(
    f"default (skip_in=64)   : wt={wt_skip(default)[0]:>6} bytes, "
    f"skip={wt_skip(default)[1]:>5} bytes"
)

# Explicit: skip path has narrower input (e.g. a lower-res branch).
narrow = kernels.conv2dk1_skip_init(
    input_channels=64,
    output_channels=256,
    skip_input_channels=32,
)
print(
    f"narrow  (skip_in=32)   : wt={wt_skip(narrow)[0]:>6} bytes, "
    f"skip={wt_skip(narrow)[1]:>5} bytes"
)

# Explicit: skip path is wider.
wide = kernels.conv2dk1_skip_init(
    input_channels=64,
    output_channels=256,
    skip_input_channels=128,
)
print(
    f"wide    (skip_in=128)  : wt={wt_skip(wide)[0]:>6} bytes, "
    f"skip={wt_skip(wide)[1]:>5} bytes"
)

print("\n  weights = (input_channels + skip_input_channels) * output_channels")
print("  skip    = input_width * skip_input_channels")

default (skip_in=64)   : wt= 32768 bytes, skip= 2048 bytes
narrow  (skip_in=32)   : wt= 24576 bytes, skip= 1024 bytes
wide    (skip_in=128)  : wt= 49152 bytes, skip= 4096 bytes

  weights = (input_channels + skip_input_channels) * output_channels
  skip    = input_width * skip_input_channels


In [8]:
default = kernels.conv2dk3(input_channels=64, output_channels=64)
shared = kernels.conv2dk3(
    input_channels=64,
    output_channels=32,
    weight_output_channels=64,
)


def wt_out(k):
    return k.arg_types()[3].__args__[0][0], k.arg_types()[4].__args__[0][0]


print(
    f"conv2dk3 defaults      : wt={wt_out(default)[0]:>5}, out={wt_out(default)[1]:>5}"
)
print(f"conv2dk3 shared-buffer : wt={wt_out(shared)[0]:>5}, out={wt_out(shared)[1]:>5}")
print("  (two workers share the 36864-byte weights, each writes 1024 bytes)")

conv2dk3 defaults      : wt=36864, out= 2048
conv2dk3 shared-buffer : wt=36864, out= 1024
  (two workers share the 36864-byte weights, each writes 1024 bytes)


## 3. `Compile[T]` parameters: three flavours

Compile-time parameters use `Compile[type]` and **must be keyword-only**
(place a `*,` before them). Three ways to bind them, mirroring Triton:

In [9]:
@jit  # A: bare decorator, params at call time
def gemm_a(a: In, b: In, c: Out, *, M: Compile[int], N: Compile[int]):
    pass


@jit(M=512, N=512)  # B: pre-bound params at decoration
def gemm_b(a: In, b: In, c: Out, *, M: Compile[int], N: Compile[int]):
    pass


@jit  # C: defaults in signature
def gemm_c(a: In, b: In, c: Out, *, M: Compile[int] = 256, N: Compile[int] = 256):
    pass


print("a:", gemm_a.compilable.compile_kwargs)
print("b:", gemm_b.compilable.compile_kwargs)
print("c:", gemm_c.compilable.compile_kwargs, "(uses signature defaults at lower-time)")

a: {}
b: {'M': 512, 'N': 512}
c: {} (uses signature defaults at lower-time)


/tmp/ipykernel_64660/547638552.py:2: UserWarning: 'gemm_a' has Compile[T] parameters with no defaults and no pre-bound values: ['M', 'N'].
  You must pass these as keyword arguments at every call:
    kernel(..., M=..., N=...)
  Omitting them will raise TypeError at compile time.
  def gemm_a(a: In, b: In, c: Out, *, M: Compile[int], N: Compile[int]):


Unknown kwargs to `@iron.jit` warn; non-keyword-only `Compile[T]`
params raise `TypeError` with a fix-it suggestion.

In [10]:
import warnings

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")

    @jit(NOPE=99)
    def oops(x: In, y: Out, *, N: Compile[int]):
        pass

    print("warning:", w[-1].message)

  Valid Compile[T] params: ['N'].
  Config keys: ['aiecc_flags', 'compile_flags', 'include_paths', 'object_files', 'source_files', 'trace_config', 'use_cache'].


## 4. `lower()` vs `specialize()` vs `compile()` vs `__call__`

A `CallableDesign` exposes four entry points with very different costs.

| call            | does                                          | cost      |
|-----------------|-----------------------------------------------|-----------|
| `.specialize(**kw)` | bind compile params, return new CallableDesign | µs        |
| `.lower(**kw)`  | run generator → MLIR string                   | ms        |
| `.compile()`    | full aiecc → cached `final.xclbin` + `insts.bin` | s (then cached) |
| `design(*tensors, **kw)` | compile (cached) + run on NPU       | ms after first |


In [11]:
t = time.perf_counter()
spec = passthrough.specialize(N=4096)
print(f"specialize()  : {(time.perf_counter() - t) * 1e6:6.0f} µs")

t = time.perf_counter()
mlir = spec.lower()
print(
    f"lower()       : {(time.perf_counter() - t) * 1e3:6.1f} ms  ({len(mlir):,} chars)"
)

t = time.perf_counter()
xclbin, insts = spec.compile()
print(f"compile()     : {(time.perf_counter() - t) * 1e3:6.1f} ms  (cache hit)")
print(f"  xclbin -> {xclbin}")
print(f"  insts  -> {insts}")

specialize()  :     39 µs
lower()       :   11.1 ms  (2,230 chars)
compile()     :   12.5 ms  (cache hit)
  xclbin -> /home/ehunhoff/.npu/cache/468342b8eb578f11761bb59d/final.xclbin
  insts  -> /home/ehunhoff/.npu/cache/468342b8eb578f11761bb59d/insts.bin


## 5. Content-addressed caching

Every compiled design lands in `~/.npu/cache/<hash>/`. The hash is
content-addressed on:

- the generator function's bytecode + closure
- the resolved `compile_kwargs` (sorted, JSON-stable)
- the target arch (`aie2` / `aie2p`)
- Peano's `clang++` mtime (so a Peano upgrade auto-invalidates)
- aiecc's mtime
- any `source_files` mtimes (so a `.cc` edit auto-invalidates)

Re-running an unchanged design is ~free; changing *anything* triggers
a fresh compile.

In [12]:
from aie.utils.compile import NPU_CACHE_HOME

print(f"Cache root: {NPU_CACHE_HOME}")
print(f"Cache entries on this machine: {len(list(NPU_CACHE_HOME.glob('*')))}")
print(f"\npassthrough(N=4096) artifacts:")
for p in sorted(Path(xclbin).parent.iterdir()):
    print(f"  {p.name}")

Cache root: /home/ehunhoff/.npu/cache
Cache entries on this machine: 54

passthrough(N=4096) artifacts:
  .lock
  aie.mlir
  final.xclbin
  input_with_addresses.mlir
  insts.bin
  main.pdi
  main_aie_cdo_elfs.bin
  main_aie_cdo_enable.bin
  main_aie_cdo_init.bin
  main_aie_partition.json
  main_core_0_2.elf
  main_core_0_2.ld.script
  main_core_0_2.ll
  main_core_0_2.o
  main_core_0_2.opt.ll
  main_core_0_2.peanohack.ll
  main_design.bif
  main_kernels.json
  main_mem_topology.json


### New: hashing tolerates missing Peano

Hashing only needs a *stable identity* for cache lookup and dict
membership — it does not need to actually compile anything. A recent
fix lets `hash(design)` succeed even when Peano isn't installed, by
catching the `RuntimeError` raised by `peano_install_dir()` and
falling back to an `"absent"` sentinel for the Peano hash component.

In [13]:
import aie.compiler.aiecc.configure as _aiecc

real_dir = _aiecc.peano_install_dir
_aiecc.peano_install_dir = "peano_not_found"
try:
    h = hash(passthrough.compilable)
    print(f"hash(design) without Peano: {h}  (no crash!)")
finally:
    _aiecc.peano_install_dir = real_dir
print(f"hash(design) with Peano   : {hash(passthrough.compilable)}")

hash(design) without Peano: -6207926861777251518  (no crash!)
hash(design) with Peano   : 1685630278334112425


## 6. Tensor-arg validation

When you call a JIT design, the framework parses `aie.dma_bd` ops out
of the lowered MLIR and checks that each runtime tensor's element count
matches the total bytes the kernel will DMA into / out of that argument.
Validation runs on both fresh compiles and cache hits, so wrong-size
tensors fail loudly *before* the NPU runs:

In [14]:
wrong_size = iron.zeros(99, dtype=np.uint8)
try:
    passthrough(wrong_size, out_t, N=4096)
except RuntimeError as e:
    print("Caught expected error:")
    print(f"  {e}")

Caught expected error:
  Tensor argument 'x' has 99 elements but the kernel was compiled for 4096 elements.
Compile[T] parameters used at compile time: {'N': 4096}


### New: multi-DMA fan-out

Some designs split one host buffer across N AIE columns where each
column gets a *different-sized* slice (e.g. resnet's weights buffer:
73728 + 69632 + 69632 = 212992 bytes). Validation now sums per-host-arg
DMA sizes, so the per-arg total is checked against the tensor element
count — handles single-DMA and multi-DMA cases uniformly.

In [15]:
# `parse_dma_sizes` is a private helper exposed here just so we can show
# the per-arg totals.  After a `compile()` (or any call), the same list
# is also stored on the design at `compilable._expected_tensor_sizes`.
from aie.utils.compile.jit._dma_size_parser import parse_dma_sizes

cache_dir = Path(xclbin).parent
sizes = parse_dma_sizes(cache_dir)
print(f"Per-host-arg DMA totals for passthrough(N=4096): {sizes}")
print(f"  arg[0] (in) : {sizes[0]} bytes")
print(f"  arg[1] (out): {sizes[1]} bytes")

Per-host-arg DMA totals for passthrough(N=4096): [4096, 4096]
  arg[0] (in) : 4096 bytes
  arg[1] (out): 4096 bytes


## 7. Helper utilities cheat-sheet

A round-up of the helpers that landed on this branch — what to import
and when to reach for them.

| Helper | What it does |
|--------|--------------|
| `aie.iron.tensor / zeros / ones / rand / randint / arange` | Build host tensors targeting `device="npu"` (or `"cpu"`). |
| `aie.iron.{In, Out, InOut, Compile}` | Type annotation markers for `@jit` generator params. |
| `aie.iron.{jit, CompilableDesign, CallableDesign}` | The JIT decorator + the two design wrapper classes. |
| `aie.iron.kernels.*` | Pre-packaged kernel factories — see §2. |
| `aie.iron.{compile_context, get_compile_arg}` | Inject compile-time values into generators that aren't kwargs of the top-level function (advanced). |
| `aie.utils.benchmark.{run_iters, print_benchmark}` | Warmup + timed iters → stats. |
| `aie.utils.verify.{nearly_equal, count_mismatches}` | Sloppy-equality compare for LUT/saturating kernel outputs. |
| `aie.utils.trace.TraceConfig` | Hardware trace plumbing — see §9. |
| `aie.utils.trace.utils.print_cycles_summary` | Pretty-print per-event cycle counts from a parsed trace. |
| `aie.utils.compile.NPU_CACHE_HOME` | Cache root path (default `~/.npu/cache`; override with `$NPU_CACHE_HOME`). |
| `aie.utils.compile.jit._dma_size_parser.parse_dma_sizes` | Per-host-arg DMA totals from `input_with_addresses.mlir`. |
| `aie.utils.set_current_device` | Set the device used by `iron.get_current_device()` and JIT compile. |
| `aie.utils.DefaultNPURuntime` | Cached XRT runtime — auto-picks NPU1/NPU2 from the device's product name. |

In [16]:
# `compile_context` — inject compile args into nested helpers
from aie.iron import compile_context, get_compile_arg


def make_fifo_pair(line_ty):
    # This helper doesn't take a `name_prefix` arg, but it can read one
    # from the active CompileContext.
    prefix = get_compile_arg("prefix", default="")
    return ObjectFifo(line_ty, name=f"{prefix}in"), ObjectFifo(
        line_ty, name=f"{prefix}out"
    )


with compile_context(prefix="layer1_"):
    line_ty = np.ndarray[(64,), np.dtype[np.uint8]]
    of_in, of_out = make_fifo_pair(line_ty)
    print("Inside context:", of_in.name, of_out.name)

# Outside the context, the default kicks in.
of_in, of_out = make_fifo_pair(line_ty)
print("Outside context:", of_in.name, of_out.name)

Inside context: layer1_in layer1_out
Outside context: in out


## 8. Tracking compilation: log every command the JIT runs

The compile path uses `logging.getLogger("aie.utils.compile")`. Set
its level to `DEBUG` and you'll see:

- **`compile_external_kernel`** (in `aie.utils.compile.utils`): the
  `clang++` invocation for each `ExternalFunction` from the kernel
  library — its command line and stdout.
- **`compile_mlir_module`** (same logger): the captured stdout/stderr
  from the `aiecc` subprocess (which itself fans out to `xclbinutil`,
  `bootgen`, etc.).
- **`compilabledesign`**: cache-hit / cache-miss messages with the
  content hash.

In [17]:
logging.basicConfig(level=logging.WARNING, format="%(name)s | %(message)s", force=True)
logging.getLogger("aie.utils.compile").setLevel(logging.DEBUG)


# `use_cache=False` skips the cache lookup so this design recompiles
# every call — handy when you want to actually see the compile commands.
# Using `kernels.passthrough(...)` here so you also see the clang
# invocation for the external kernel, not just the aiecc subprocess.
@jit(use_cache=False)
def passthrough_logged(x: In, y: Out, *, N: Compile[int]):
    line_size = N // 4
    line_ty = np.ndarray[(line_size,), np.dtype[np.uint8]]
    of_in = ObjectFifo(line_ty, name="in")
    of_out = ObjectFifo(line_ty, name="out")

    pass_through = kernels.passthrough(tile_size=line_size, dtype=np.uint8)

    def core_fn(of_in, of_out, pass_through):
        elem_in = of_in.acquire(1)
        elem_out = of_out.acquire(1)
        pass_through(elem_in, elem_out, line_size)
        of_in.release(1)
        of_out.release(1)

    worker = Worker(core_fn, [of_in.cons(), of_out.prod(), pass_through])
    rt = Runtime()
    with rt.sequence(line_ty, line_ty) as (a_in, b_out):
        rt.start(worker)
        rt.fill(of_in.prod(), a_in)
        rt.drain(of_out.cons(), b_out, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


print("Compiling passthrough_logged(N=2048) — watch for clang/aiecc:")
xclbin_logged, _ = passthrough_logged.specialize(N=2048).compile()

/tmp/ipykernel_64660/3710354340.py:10: UserWarning: 'passthrough_logged' has Compile[T] parameters with no defaults and no pre-bound values: ['N'].
  You must pass these as keyword arguments at every call:
    kernel(..., N=...)
  Omitting them will raise TypeError at compile time.
  def passthrough_logged(x: In, y: Out, *, N: Compile[int]):
aie.utils.compile.jit.compilabledesign | Cache miss for 'passthrough_logged' (hash=f130ba9bde32bdc5ae38d747); compiling...


aie.utils.compile.utils | Compiling with: /scratch/ehunhoff/mlir-aie/ironenv/lib/python3.10/site-packages/llvm-aie/bin/clang++ /home/ehunhoff/.npu/cache/f130ba9bde32bdc5ae38d747/passThroughLine.cc -c -o /home/ehunhoff/.npu/cache/f130ba9bde32bdc5ae38d747/passThroughLine.o -I/scratch/ehunhoff/mlir-aie/install/include -std=c++20 -Wno-parentheses -Wno-attributes -Wno-macro-redefined -Wno-empty-body -O2 -DNDEBUG --target=aie2-none-unknown-elf -I /scratch/ehunhoff/mlir-aie/install/include -I /scratch/ehunhoff/mlir-aie/install/include/aie_kernels/generic -DBIT_WIDTH=8


Compiling passthrough_logged(N=2048) — watch for clang/aiecc:


aie.utils.compile.utils | 

****** Bootgen v2023.2
  **** Build date : May  8 2026-15:54:43
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.


[INFO]   : Bootimage generated successfully

XRT Build Version: 2.19.0 (HEAD)
       Build Date: 2025-01-14 09:00:07
          Hash ID: acc144998d650acbfda7e5919a1290de8f8c7735
Creating a default 'in-memory' xclbin image.

Section: 'MEM_TOPOLOGY'(6) was successfully added.
Size   : 88 bytes
Format : JSON
File   : '/home/ehunhoff/.npu/cache/f130ba9bde32bdc5ae38d747/main_mem_topology.json'

Section: 'AIE_PARTITION'(32) was successfully added.
Size   : 2536 bytes
Format : JSON
File   : '/home/ehunhoff/.npu/cache/f130ba9bde32bdc5ae38d747/main_aie_partition.json'
Info: Embedded Metadata section is missing project.platform.device.core element, adding it.
Successfully wrote (8568 bytes) to the output file: /home/ehunhoff/.npu/cache/f130ba9bde32bdc5ae38d747/final.xc

The kernel directory holds **everything** aiecc produced — useful
when you need to inspect the lowered MLIR, the post-placement IR, or
the actual `.o` files that linked into the final xclbin:

In [18]:
kernel_dir = Path(xclbin_logged).parent
print(f"Cache dir: {kernel_dir}\n")
for p in sorted(kernel_dir.iterdir())[:20]:
    size = p.stat().st_size if p.is_file() else "<dir>"
    print(f"  {p.name:<40} {size}")

Cache dir: /home/ehunhoff/.npu/cache/f130ba9bde32bdc5ae38d747

  .lock                                    0
  aie.mlir                                 2182
  final.xclbin                             8568
  input_with_addresses.mlir                7061
  insts.bin                                300
  main.pdi                                 2112
  main_aie_cdo_elfs.bin                    992
  main_aie_cdo_enable.bin                  44
  main_aie_cdo_init.bin                    984
  main_aie_partition.json                  711
  main_core_0_2.elf                        2108
  main_core_0_2.ld.script                  1243
  main_core_0_2.ll                         3531
  main_core_0_2.o                          1140
  main_core_0_2.opt.ll                     2529
  main_core_0_2.peanohack.ll               3443
  main_design.bif                          361
  main_kernels.json                        1851
  main_mem_topology.json                   400
  passThroughLine.cc                

In [19]:
# Quiet down the logger again so the next sections aren't noisy.
logging.getLogger("aie.utils.compile").setLevel(logging.WARNING)
print("Logging restored to WARNING.")

Logging restored to WARNING.


## 9. Hardware trace via `TraceConfig`

The hardware trace machinery threads through `@iron.jit` in two steps:

1. **Generator side**: declare `trace_config: Compile[TraceConfig | None] = None`,
   wrap your `Worker(...)` with `trace=1 if trace_config else 0`,
   and call `rt.enable_trace(trace_config.trace_size, workers=[...])`
   inside the runtime sequence.

2. **Caller side**: pass `trace_config=TraceConfig(trace_size=N)` at
   call time. After the call, `trace_config.physical_mlir_path` is
   populated so you can run `trace_config.trace_to_json(...)` to parse
   `trace.txt` into a Chrome-tracing JSON.

This mirrors what `programming_examples/basic/passthrough_kernel/passthrough_kernel.py`
does end-to-end.

In [20]:
from aie.utils.trace import TraceConfig


@jit
def passthrough_with_trace(
    x: In,
    y: Out,
    *,
    N: Compile[int],
    trace_config: Compile[TraceConfig | None] = None,
):
    line_ty = np.ndarray[(N,), np.dtype[np.uint8]]
    of_in = ObjectFifo(line_ty, name="in")
    of_out = ObjectFifo(line_ty, name="out")

    def core_fn(of_in, of_out):
        elem_in = of_in.acquire(1)
        elem_out = of_out.acquire(1)
        for i in range_(N):
            elem_out[i] = elem_in[i]
        of_in.release(1)
        of_out.release(1)

    # Wire trace=1 onto each worker we want to instrument.
    worker = Worker(
        core_fn, [of_in.cons(), of_out.prod()], trace=1 if trace_config else 0
    )

    rt = Runtime()
    with rt.sequence(line_ty, line_ty) as (a_in, b_out):
        if trace_config:
            rt.enable_trace(trace_config.trace_size, workers=[worker])
        rt.start(worker)
        rt.fill(of_in.prod(), a_in)
        rt.drain(of_out.cons(), b_out, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


# `lower()` runs the generator and shows the MLIR — confirm the trace
# wiring (`packetflow`s into the trace ddr buffer) was actually emitted.
trace_cfg = TraceConfig(trace_size=8192)
mlir_with_trace = passthrough_with_trace.lower(N=4096, trace_config=trace_cfg)
trace_lines = [
    l for l in mlir_with_trace.splitlines() if "trace" in l.lower() or "packetflow" in l
]
print(f"{len(trace_lines)} trace-related ops in the lowered MLIR:")
for line in trace_lines[:6]:
    print("  " + line.strip()[:100])

17 trace-related ops in the lowered MLIR:
  aie.trace @trace_core_1(%logical_core) {
  aie.trace.mode "Event-Time"
  aie.trace.packet id = 1 type = core
  aie.trace.event <"INSTR_EVENT_0">
  aie.trace.event <"INSTR_EVENT_1">
  aie.trace.event <"INSTR_VECTOR">


/tmp/ipykernel_64660/1357409869.py:5: UserWarning: 'passthrough_with_trace' has Compile[T] parameters with no defaults and no pre-bound values: ['N'].
  You must pass these as keyword arguments at every call:
    kernel(..., N=...)
  Omitting them will raise TypeError at compile time.
  def passthrough_with_trace(


**Calling it on hardware** then writes a `trace.txt` next to your
cwd; `trace_config.physical_mlir_path` is auto-populated after the
call so `trace_config.trace_to_json(...)` can parse it, and
`aie.utils.trace.utils.print_cycles_summary` gives you the per-event
cycle breakdown — useful for spotting which kernel call is the
bottleneck.

The runnable end-to-end (driver + trace parse + summary) lives in
**`programming_examples/basic/passthrough_kernel/passthrough_kernel.py`**:

```bash
cd programming_examples/basic/passthrough_kernel
make trace        # equivalent to: python3 passthrough_kernel.py -t 8192
```

(We don't run it inline here — Jupyter and the XRT trace runtime have
some interaction that makes `kernel(...)` with `trace_config` flaky in
notebooks even though it's solid as a standalone script.)

## 10. `aie.utils.benchmark` — timing + warmup

`run_iters(fn, *args, warmup=N, iters=M)` runs the design `M+N` times
(discarding the first `N`) and returns a `BenchmarkResult` with
`avg_us` / `min_us` / `max_us` for both end-to-end host latency and
NPU-only time.  `print_benchmark(result)` formats it nicely.

In [21]:
from aie.utils.benchmark import run_iters, print_benchmark

bench = run_iters(passthrough, in_t, out_t, N=4096, warmup=2, iters=10)
print_benchmark(bench)

NPU time     (avg/min/max us): 177.1 / 168.8 / 184.6
End-to-end   (avg/min/max us): 286.6 / 275.1 / 311.0


## 11. `aie.utils.verify` — sloppy equality for AIE outputs

NPU kernels are often LUT approximations or use saturating arithmetic,
so exact `np.array_equal` is too strict. `count_mismatches(actual, ref,
rtol=0.128, atol=None)` returns `(n_errors, n_checked)` — defaults
to 12.8% relative tolerance and stops at the first inf / nan.
`nearly_equal(...)` returns the boolean mask if you need it.

In [22]:
from aie.utils.verify import count_mismatches, nearly_equal

ref = np.linspace(0, 10, 1000, dtype=np.float32)
approx = ref + np.random.uniform(-0.05, 0.05, size=ref.shape).astype(np.float32)
errors, n = count_mismatches(approx, ref, rtol=0.05)
print(f"count_mismatches(rtol=0.05): {errors} / {n} samples outside tolerance")

mask = nearly_equal(approx, ref, rtol=0.05)
print(f"nearly_equal: {int(mask.sum())} / {mask.size} samples within tolerance")

count_mismatches(rtol=0.05): 30 / 1000 samples outside tolerance
nearly_equal: 970 / 1000 samples within tolerance


## 12. Putting it all together — full case studies

Single-file `@iron.jit` designs that exercise the patterns above
end-to-end:

| Design | Path |
|--------|------|
| Passthrough kernel + trace | `programming_examples/basic/passthrough_kernel/passthrough_kernel.py` |
| Vector exp (LUT-backed)    | `programming_examples/basic/vector_exp/vector_exp.py` |
| GEMM (parametrized AOT)    | `programming_examples/basic/matrix_multiplication/single_core/03_matmul/` |
| Vector reduce, eltwise add/mul, vector scalar mul | `programming_examples/basic/vector_*/`, `programming_examples/basic/eltwise_*/` |
| ResNet conv2_x | `programming_examples/ml/resnet/layers_conv2_x/resnet.py` |

The resnet design is the most complete demo of shared-buffer kernel
factories + multi-DMA fan-out: it streams weights to 3 columns at
different sizes and uses both
`conv2dk1_skip_init(skip_input_channels=...)` and
`conv2dk3(weight_output_channels=...)`.

## ✨ That's the tour!

Every cell here ran on a live Phoenix NPU. The same notebook works
unchanged on Strix (NPU2) — the device auto-detection in §0 handles
both.